In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from python.functions.bridge import seasonal_factors

RAW_DIR = "../../data/raw"
OUT_DIR = "../../data/python_master"

# The models deflate by the GDP deflator (01_data_loading.R builds
# lrprc = log(hprice/gdp_def), lrcc = log(cc/gdp_def)), so the OBR real paths
# have to use the same deflator. EFO table 1.7 carries a GDP deflator level
# index next to CPI; "CPI.1"/"GDP deflator.1" are the level columns, the
# unsuffixed ones are year-on-year growth.
DEFLATOR = "GDP deflator.1"

# Seasonal factors are estimated from 2000 onward: transactions seasonality
# shifted with the Land Registry data and the pre-2000 Meen splice.
SEAS_SINCE = 2000

In [ ]:
with pd.ExcelFile(f"{RAW_DIR}/OBR/efo-march-2026-detailed-forecast-tables-economy.xlsx") as xls:
    hs_raw = pd.read_excel(xls, sheet_name='1.16', skiprows=2, index_col=1,
                       nrows=93)
    rate_raw = pd.read_excel(xls, sheet_name='1.9', skiprows=2, index_col=1,
                         nrows = 93)
    defl_raw = pd.read_excel(xls, sheet_name='1.7', skiprows=3, index_col=1,
                        nrows=93)



for df in [hs_raw, rate_raw, defl_raw]:
    df.drop(columns="Unnamed: 0", inplace=True)

hs = hs_raw.rename(columns={
    'House price index \n(Jan 2023 = 100)': "hprice",
    'Residential property transactions \n(000s, seasonally adjusted)': "vol",
})[["hprice", "vol"]]
rate = rate_raw.rename(columns={"Bank Rate": "r3"})["r3"]
defl = defl_raw[DEFLATOR].rename("defl")

proc = pd.concat([hs, rate, defl], axis=1).reset_index().rename(columns={"index": "period"})
proc['period'] = pd.PeriodIndex(proc['period'], freq='Q')

# Filtering to observation period end - CHANGE THIS WHEN NEW DATA RELEASED
proc = proc[proc["period"] >= "2025Q4"].reset_index(drop=True)

# Parsing construction costs
url = "https://costmodelling.com/construction-indices"
cc_html = BeautifulSoup(requests.get(url).text, "html.parser")
rows = cc_html.find("table", class_="indices").find_all("tr")[1:]
cc = pd.DataFrame([[c.text.strip() for c in r.find_all('td')[:3]] for r in rows],
                  columns=["date", "tpi", "bci"])
cc['date'] = cc['date'].replace('', pd.NA).ffill()
cc['quarter'] = cc.groupby('date').cumcount() + 1
cc['period'] = pd.PeriodIndex(cc['date'] + 'Q' + cc['quarter'].astype(str), freq='Q')
cc['bci'] = pd.to_numeric(cc['bci'], errors='coerce')
proc = proc.merge(cc[["period", "bci"]], on='period', how='left')

# Beyond last real bci hold the index flat in real terms, i.e. grow it with the
# same deflator the model uses -- otherwise lrcc drifts on a CPI/GDP-deflator wedge
last_real = proc['bci'].last_valid_index()
for i in range(last_real + 1, len(proc)):
    proc.loc[i, 'bci'] = proc.loc[i-1, 'bci'] * (proc.loc[i, 'defl'] / proc.loc[i-1, 'defl'])


# England history, for the jump-off anchors and the seasonal factors
eng = pd.read_csv("../../data/python_master/england_master.csv", index_col=0)
eng.index = pd.PeriodIndex(eng.index, freq="Q")

# Taking growth paths
proc["dlrprc"] = np.log(proc["hprice"]).diff() - np.log(proc["defl"]).diff()
proc["dlvol"]  = np.log(proc["vol"]).diff()
proc["dlrcc"]  = np.log(proc["bci"]).diff()    - np.log(proc["defl"]).diff()

# Taking last observations 2025Q4 in england data
last = eng.loc["2025Q4"]
lrprc_anchor = np.log(last['hprice'] / last['gdp_def'])
lvol_anchor  = np.log(last['vol'])
lrcc_anchor  = np.log(last['cc']    / last['gdp_def'])

# OBR publishes transactions seasonally adjusted, but `vol` in the master is
# raw Land Registry counts and the ARDL/NARDL/VECM were estimated on that NSA
# series (long-run lvol elasticity 0.83). Splicing an SA growth path onto an
# NSA anchor would both strip the seasonal and carry the anchor quarter's own
# seasonal (Q4: +0.07 log points) through the whole horizon as a level shift.
# So: de-seasonalise the anchor, accumulate the SA path, add the seasonal back.
vol_seas = seasonal_factors(eng["vol"], since=SEAS_SINCE)
q = proc["period"].dt.quarter
print("log-vol seasonal factors:", vol_seas.round(4).to_dict())

# House prices carry no SA flag in table 1.16 and the measured NSA seasonal in
# real house prices is ~1 per cent, so that path is left as published.
proc["lrprc"] = lrprc_anchor + proc["dlrprc"].fillna(0).cumsum()
proc["lvol"]  = ((lvol_anchor - vol_seas[4]) + proc["dlvol"].fillna(0).cumsum()
                 + q.map(vol_seas).to_numpy())
proc["lrcc"]  = lrcc_anchor  + proc["dlrcc"].fillna(0).cumsum()

# The 2025Q4 row is the jump-off anchor and is dropped downstream; it must
# reproduce the master exactly, seasonal round-trip included.
assert np.isclose(proc.loc[0, "lvol"], lvol_anchor), "lvol anchor not preserved"
assert proc.loc[0, "period"] == pd.Period("2025Q4", freq="Q")

out = proc[["period", "lrprc", "lvol", "r3", "lrcc"]]
out.to_csv(f"{OUT_DIR}/OBR/obr_scenario.csv", index=False)

In [ ]:
out